# 02 — Document Intelligence smoke test

Uploads a sample PDF, runs `prebuilt-layout`, prints the first text chunk, any tables found, and extracts embedded images — each image is uploaded to Blob and described by GPT-4o vision so it can be retrieved by the RAG layer and shown to users.

**Place** any PDF at `notebooks/sample.pdf` before running.

In [ ]:
import sys, os, pathlib
sys.path.insert(0, os.path.abspath('..'))
from src.blob_client import BlobService
from src.doc_intelligence import DocIntelService
from src.openai_client import OpenAIService

PDF_PATH = pathlib.Path('Multi_Agent_Research_System_Architecture.pdf')
assert PDF_PATH.exists(), 'Drop a sample.pdf next to this notebook'

blob = BlobService()
blob.ensure_container()
name = 'docmind-test/sample.pdf'
pdf_bytes = PDF_PATH.read_bytes()
blob.upload(name, pdf_bytes, content_type='application/pdf')
url = blob.get_url(name)
print('PDF URL:', url[:80], '…')

INFO: Incomplete environment configuration for EnvironmentCredential. These variables are set: AZURE_TENANT_ID
INFO: ManagedIdentityCredential will use IMDS
INFO: Request URL: 'https://pdgrag.blob.core.windows.net/sgp-dload05s-ra-input?restype=REDACTED&sp=REDACTED&st=REDACTED&se=REDACTED&spr=REDACTED&sv=REDACTED&sr=REDACTED&sig=REDACTED'
Request method: 'PUT'
Request headers:
    'x-ms-version': 'REDACTED'
    'Accept': 'application/xml'
    'User-Agent': 'azsdk-python-storage-blob/12.28.0 Python/3.14.3 (Windows-11-10.0.26200-SP0)'
    'x-ms-date': 'REDACTED'
    'x-ms-client-request-id': 'f550ac29-4acf-11f1-ba99-bcd22c162bea'
No body was attached to the request
INFO: Response status: 403
Response headers:
    'Content-Length': '246'
    'Content-Type': 'application/xml'
    'Server': 'Windows-Azure-Blob/1.0 Microsoft-HTTPAPI/2.0'
    'x-ms-request-id': 'e6276635-401e-0032-51dc-de558a000000'
    'x-ms-client-request-id': 'f550ac29-4acf-11f1-ba99-bcd22c162bea'
    'x-ms-version': 'REDAC

PDF URL: https://pdgrag.blob.core.windows.net/sgp-dload05s-ra-input/docmind-test/sample.p …


In [2]:
doc_intel = DocIntelService()
result = doc_intel.extract_pdf(url)
print('pages:', result['pages'])
print('text chunks:', len(result['text_chunks']))
print('tables:', len(result['tables']))
print('--- first 600 chars of page 1 ---')
print(result['text_chunks'][0]['content'][:600] if result['text_chunks'] else '(empty)')

INFO: Document Intelligence analysing https://pdgrag.blob.core.windows.net/sgp-dload05s-ra-input/docmind-test/sample.p
INFO: Request URL: 'https://azdocumentai.cognitiveservices.azure.com//documentintelligence/documentModels/prebuilt-layout:analyze?api-version=2024-11-30&outputContentFormat=REDACTED'
Request method: 'POST'
Request headers:
    'content-type': 'application/json'
    'Content-Length': '237'
    'Accept': 'application/json'
    'x-ms-client-request-id': '00dc1527-4ad0-11f1-ad26-bcd22c162bea'
    'User-Agent': 'azsdk-python-ai-documentintelligence/1.0.2 Python/3.14.3 (Windows-11-10.0.26200-SP0)'
    'Ocp-Apim-Subscription-Key': 'REDACTED'
A body is sent with the request
INFO: Response status: 202
Response headers:
    'Content-Length': '0'
    'Operation-Location': 'REDACTED'
    'x-envoy-upstream-service-time': 'REDACTED'
    'apim-request-id': 'REDACTED'
    'Strict-Transport-Security': 'REDACTED'
    'x-content-type-options': 'REDACTED'
    'x-ms-region': 'REDACTED'
   

pages: 27
text chunks: 26
tables: 24
--- first 600 chars of page 1 ---
Multi-Agent Research System
Architecture Design & Technical Documentation
Author
Kumar Nishant - Senior Al Engineer
Team
MLEDSI
Manager
Krutika Kansara
Date
April 20, 2026
Version
1.2
Status
Draft - For Review


In [3]:
if result['tables']:
    print(result['tables'][0]['content'][:1000])
else:
    print('No tables detected.')

| Author | Kumar Nishant - Senior Al Engineer |
| --- | --- |
| Team | MLEDSI |
| Manager | Krutika Kansara |
| Date | April 20, 2026 |
| Version | 1.2 |
| Status | Draft - For Review |


In [ ]:
result.keys()

## What's in the result?

`extract_pdf` returns a dict with these keys:

| Key | Type | What it is |
|---|---|---|
| `pages` | int | Page count |
| `text_chunks` | list | Legacy per-page raw text (one entry per page) |
| `paragraphs` | list | Paragraphs in reading order (`id`, `content`, `page`, `bbox`, `role`, `section_id`) |
| `sections` | list | Hierarchical sections (`heading`, `level`, `path`, `parent_id`, child ids) |
| `tables` | list | Markdown-rendered tables with caption + neighbor paragraphs |
| `figures` | list | Raw DI figure objects (consumed by `extract_images`) |
| `figures_meta` | list | Section / neighbor metadata keyed by figure id |

### Paragraphs (reading order)

Each paragraph carries its `page`, bounding box, optional `role` (e.g. `title`, `sectionHeading`, `pageHeader`), and the `section_id` it belongs to. Roles are how we later identify section headings.

In [ ]:
from collections import Counter

paragraphs = result['paragraphs']
print(f'total paragraphs: {len(paragraphs)}')
print('roles:', Counter(p['role'] for p in paragraphs))
print()
print('--- first 5 paragraphs ---')
for p in paragraphs[:5]:
    role = p['role'] or '-'
    bbox = [round(v, 1) for v in p['bbox']] if p['bbox'] else None
    print(f"  {p['id']:>4}  page={p['page']}  role={role:<14}  section={p['section_id']}")
    print(f"        bbox={bbox}")
    print(f"        {p['content'][:120]!r}")
    print()

total paragraphs: 511
roles: Counter({None: 444, <ParagraphRole.SECTION_HEADING: 'sectionHeading'>: 61, <ParagraphRole.TITLE: 'title'>: 3, <ParagraphRole.PAGE_FOOTER: 'pageFooter'>: 3})

--- first 5 paragraphs ---
    p0  page=1  role=ParagraphRole.TITLE  section=s1
        bbox=[122.8, 251.7, 486.2, 303.4]
        'Multi-Agent Research System Architecture Design & Technical Documentation'

    p1  page=1  role=-               section=None
        bbox=[72.0, 340.7, 305.9, 354.7]
        'Author'

    p2  page=1  role=-               section=None
        bbox=[305.4, 340.7, 539.3, 354.7]
        'Kumar Nishant - Senior Al Engineer'

    p3  page=1  role=-               section=None
        bbox=[72.0, 354.2, 305.9, 368.7]
        'Team'

    p4  page=1  role=-               section=None
        bbox=[305.9, 354.7, 539.3, 368.7]
        'MLEDSI'



### Sections (hierarchy + breadcrumbs)

DI returns sections with element refs like `/paragraphs/12`, `/sections/3`, `/tables/0`, `/figures/1`. We resolve these into typed id lists and walk parents up to compute `level` and `path` (the heading breadcrumb used for retrieval grouping).

In [5]:
sections = result['sections']
print(f'total sections: {len(sections)}\n')
for s in sections[:15]:
    indent = '  ' * (s['level'] - 1)
    breadcrumb = ' > '.join(s['path'])
    print(f"{indent}[L{s['level']}] {s['id']}  {s['heading'][:60]!r}")
    print(f"{indent}      path: {breadcrumb}")
    print(f"{indent}      paragraphs={len(s['paragraph_ids'])}  tables={len(s['table_ids'])}  figures={len(s['figure_ids'])}  parent={s['parent_id']}")

total sections: 65

[L1] s0  'Section 0'
      path: Section 0
      paragraphs=0  tables=0  figures=0  parent=None
  [L2] s1  'Multi-Agent Research System Architecture Design & Technical '
        path: Section 0 > Multi-Agent Research System Architecture Design & Technical Documentation
        paragraphs=1  tables=1  figures=0  parent=s0
    [L3] s2  'Table of Contents'
          path: Section 0 > Multi-Agent Research System Architecture Design & Technical Documentation > Table of Contents
          paragraphs=38  tables=0  figures=0  parent=s1
  [L2] s3  '1. Executive Summary'
        path: Section 0 > 1. Executive Summary
        paragraphs=3  tables=0  figures=0  parent=s0
    [L3] s4  'Key Highlights:'
          path: Section 0 > 1. Executive Summary > Key Highlights:
          paragraphs=9  tables=0  figures=0  parent=s3
  [L2] s5  '2. Introduction'
        path: Section 0 > 2. Introduction
        paragraphs=3  tables=0  figures=0  parent=s0
    [L3] s6  '2.1 Purpose of This D

### Tables (markdown + neighbor paragraphs)

Tables are rendered to markdown by `_table_to_markdown`. Each table also carries `section_path` (where it lives in the doc) and `neighbor_paragraph_ids` — the paragraphs immediately above/below it that humans treat as its caption / explanation.

In [6]:
from IPython.display import Markdown, display

para_by_id = {p['id']: p for p in result['paragraphs']}

for t in result['tables'][:2]:
    display(Markdown(f"#### Table `{t['id']}` — page {t['page']}"))
    print('section_path :', t['section_path'])
    print('caption      :', t['caption'] or '(none)')
    print('neighbors    :', t['neighbor_paragraph_ids'])
    for pid in t['neighbor_paragraph_ids']:
        if pid in para_by_id:
            print(f"  {pid}: {para_by_id[pid]['content'][:140]!r}")
    display(Markdown(t['content']))

#### Table `t0` — page 1

section_path : Section 0 > Multi-Agent Research System Architecture Design & Technical Documentation
caption      : (none)
neighbors    : ['p0']
  p0: 'Multi-Agent Research System Architecture Design & Technical Documentation'


| Author | Kumar Nishant - Senior Al Engineer |
| --- | --- |
| Team | MLEDSI |
| Manager | Krutika Kansara |
| Date | April 20, 2026 |
| Version | 1.2 |
| Status | Draft - For Review |

#### Table `t1` — page 6

section_path : Section 0 > 3. Business Value & Usefulness > 3.1 Enterprise Use Cases
caption      : (none)
neighbors    : ['p78', 'p79']
  p78: '3.1 Enterprise Use Cases'
  p79: "The system's intent-aware, multi-agent architecture makes it applicable across a wide range of enterprise scenarios. The following table hig"


| Use Case | Intent Type | Business Value |
| --- | --- | --- |
| Market Research & Competitive Analysis | Comparative | Enables rapid competitive intelligence gathering with structured side-by-side comparisons, reducing analyst effort by 70-80%. |
| Technology Evaluation Reports | Deep Research | Produces comprehensive technology assessments for build-vs-buy decisions, vendor evaluations, and architecture reviews. |
| Internal Knowledge Base Generation | Summary / Deep Research | Automatically generates documentation from scattered internal sources, improving knowledge sharing across teams. |
| Executive Briefing & Strategy Reports | Summary | Delivers concise, decision-ready summaries for leadership, distilling complex topics into actionable insights. |
| Technical Blog & Thought Leadership | Blog | Generates publication-ready blog posts and whitepapers, accelerating content marketing and thought leadership initiatives. |
| Regulatory & Compliance Research | Deep Research | Automates the research of regulatory requirements across jurisdictions, ensuring thoroughness and reducing compliance risk. |
| Customer-Facing RFP Responses | Comparative / Summary | Generates structured, evidence- backed responses to RFPs and customer questionnaires with source citations. |
| Due Diligence & Investment Research | Deep Research | Produces thorough due diligence reports for M&A, partnerships, and investment decisions with multi- source validation. |

### Figures metadata (before raster extraction)

`figures` are the raw DI figure objects (bounding regions + caption) and `figures_meta` is our enriched lookup with section path and neighbor paragraphs. These two are passed into `extract_images` to drive the hybrid PyMuPDF pipeline below.

In [ ]:
print(f"raw DI figures      : {len(result['figures'])}")
print(f"figures_meta entries: {len(result['figures_meta'])}\n")

for fm in result['figures_meta'][:5]:
    bbox = [round(v, 1) for v in fm['bbox']] if fm['bbox'] else None
    print(f"{fm['id']}  page={fm['page']}  section={fm['section_id']}")
    print(f"      bbox={bbox}")
    print(f"      section_path: {fm['section_path']}")
    print(f"      neighbors   : {fm['neighbor_paragraph_ids']}")
    for pid in fm['neighbor_paragraph_ids']:
        if pid in para_by_id:
            print(f"        {pid}: {para_by_id[pid]['content'][:100]!r}")
    print()

raw DI figures      : 1
figures_meta entries: 1

f0  page=15  section=s22
      bbox=[104.4, 74.3, 496.9, 787.0]
      section_path: Section 0 > 6. Architecture Diagram
      neighbors   : ['p262', 'p289']
        p262: 'The following diagram illustrates the complete end-to-end flow of the Multi-Agent Research System, i'
        p289: 'Figure 1: Multi-Agent Research System - Architecture Flow Diagram'



## Extract images — hybrid DI + PyMuPDF pipeline

DI tells us **where** figures are; PyMuPDF gives us the **bytes**. We run two passes:

1. **Pass 1 — DI figures**: for each figure region returned by `prebuilt-layout`, render that region from the rasterized page with PyMuPDF (`page.get_pixmap(clip=rect, dpi=…)`). This captures vector charts and diagrams that have no embedded raster.
2. **Pass 2 — embedded rasters**: iterate `page.get_images(full=True)` and add any raster **not already covered** by a DI figure (IoU < `DEDUP_IOU`). Catches inline photos / logos DI didn't tag as figures.

Each kept image is uploaded to Blob and described by GPT-4o vision (caption-aware when DI provides one). Section + neighbor metadata from `figures_meta` rides along on every figure-source image.

In [ ]:
openai = OpenAIService()
images = doc_intel.extract_images(
    pdf_bytes,
    doc_id='docmind-test',
    blob=blob,
    openai=openai,
    figures=result['figures'],
    figures_meta=result['figures_meta'],
)

by_source = Counter(img['source'] for img in images)
print(f'extracted {len(images)} images  —  by source: {dict(by_source)}\n')
for img in images[:8]:
    print(f"- page {img['page']}  source={img['source']:<7}  {img['ext']}  "
          f"{img['size_bytes']:>7} bytes  fig={img['figure_id']}  -> {img['blob_name']}")

INFO: Request URL: 'https://pdgrag.blob.core.windows.net/sgp-dload05s-ra-input/docmind-test/figures/page15_fig0_0.png?sp=REDACTED&st=REDACTED&se=REDACTED&spr=REDACTED&sv=REDACTED&sr=REDACTED&sig=REDACTED'
Request method: 'PUT'
Request headers:
    'Content-Length': '188895'
    'x-ms-blob-type': 'REDACTED'
    'x-ms-blob-content-type': 'REDACTED'
    'x-ms-version': 'REDACTED'
    'Content-Type': 'application/octet-stream'
    'Accept': 'application/xml'
    'User-Agent': 'azsdk-python-storage-blob/12.28.0 Python/3.14.3 (Windows-11-10.0.26200-SP0)'
    'x-ms-date': 'REDACTED'
    'x-ms-client-request-id': '3da2be34-4ad0-11f1-b11c-bcd22c162bea'
A body is sent with the request
INFO: Response status: 201
Response headers:
    'Content-Length': '0'
    'Content-MD5': 'REDACTED'
    'Last-Modified': 'Fri, 08 May 2026 11:22:48 GMT'
    'ETag': '"0x8DEACF422C620DF"'
    'Server': 'Windows-Azure-Blob/1.0 Microsoft-HTTPAPI/2.0'
    'x-ms-request-id': '8053bdc8-c01e-003c-16dc-de7c3a000000'
    '

extracted 1 images  —  by source: {'figure': 1}

- page 15  source=figure   png   188895 bytes  fig=f0  -> docmind-test/figures/page15_fig0_0.png


In [9]:
from IPython.display import Image, Markdown, display

for img in images[:5]:
    display(Markdown(
        f"### Page {img['page']} — `{img['blob_name']}`  \n"
        f"*source:* `{img['source']}` &nbsp; *figure_id:* `{img['figure_id']}` &nbsp; "
        f"*section:* {img['section_path'] or '—'}"
    ))
    if img['caption']:
        display(Markdown(f"**Caption:** {img['caption']}"))
    try:
        display(Image(url=img['image_url']))
    except Exception as e:
        print('(could not render inline:', e, ')')
    if img['neighbor_paragraph_ids']:
        display(Markdown('**Neighbor paragraphs:**'))
        for pid in img['neighbor_paragraph_ids']:
            if pid in para_by_id:
                display(Markdown(f"- *{pid}* — {para_by_id[pid]['content'][:200]}"))
    display(Markdown(f"**Vision description:**\n\n{img['description']}"))

### Page 15 — `docmind-test/figures/page15_fig0_0.png`  
*source:* `figure` &nbsp; *figure_id:* `f0` &nbsp; *section:* Section 0 > 6. Architecture Diagram

**Neighbor paragraphs:**

- *p262* — The following diagram illustrates the complete end-to-end flow of the Multi-Agent Research System, including all nodes, edges, feedback loops, and conditional branching paths.

- *p289* — Figure 1: Multi-Agent Research System - Architecture Flow Diagram

**Vision description:**

This image is a flowchart depicting a process for handling user queries through a structured research and task management system. Below is a detailed description of the components and their relationships:

1. **User Query**: The process begins with a user query.

2. **Planner Node**: Receives the user query and proceeds to analyze intent and define the goal.

3. **Analyze Intent + Define Goal**: This step involves understanding the user's intent and setting a clear objective.

4. **Research Intent Classification**: Classifies the research intent into categories such as blog, comparative, deep research, or summary.

5. **Structured Plan Creation**: Develops a structured plan based on the classified intent.

6. **Generate Todos with status=pending**: Creates tasks (todos) with a pending status.

7. **Task Manager**: Manages the tasks and feeds into the Todo Selector.

8. **Todo Selector**: Selects pending tasks for processing.

9. **Delegation Layer**: Distributes tasks to various sub-agents.

10. **Sub-Agents (1, 2, N)**: Each sub-agent receives tasks and provides findings and sources.

11. **Aggregator**: Collects findings and sources from sub-agents, then merges, deduplicates, and structures the information.

12. **Context Synthesizer**: Synthesizes the context from the aggregated data.

13. **Todo Tracker**: Marks tasks as completed and stores the data.

14. **Todo Checker**: Checks if all todos are completed. If not, it loops back to the Todo Selector.

15. **Research Gap Detected**: If a research gap is detected, it loops back to the Planner Node for further analysis.

The flowchart illustrates a cyclical process with feedback loops to ensure comprehensive task management and research completion.

## Summary — what feeds the chunker

The chunker (`src/chunking.py`) consumes:

- **`paragraphs`** — text units in reading order, tagged with `section_id`.
- **`sections`** — to group chunks under a stable breadcrumb (`path`).
- **`tables`** — markdown body + `neighbor_paragraph_ids` inlined as caption context.
- **`images`** (from `extract_images`) — `description` is what gets embedded; `image_url` is what the UI renders. Figure-source images carry `section_path` and neighbor ids so retrieval over a chart hits both the visual description and the surrounding prose.

See [docs/doc-intelligence.md](../docs/doc-intelligence.md) for the full architecture.